In [4]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff
from plotly.subplots import make_subplots
from duckdb import sql as sqldf

from pathlib import Path
from datetime import datetime

In [61]:
np.set_printoptions(floatmode='fixed', precision=6,  suppress=True)

## Exponential scaling range for plotly slider for parameters with large ranges

In [62]:
def exp_scale(mn, mx, dip, num):
    """Produces an exponentially scaled axis for the slider control.
        When you need to control a variable with possible values in a wide range of orders of magnitude.
        e.g.  0.0001 < x < 10000

    Args:
        mn (float): minimal value of the scale. also defines minimal order of magnitude 
        mx (float): maximal value of the scale. also defines maximal order of magnitude
        num (int): the amount of marks to produce

    Returns:
        float, float: ax - the linear values for the slider; val - the exponential values for the parameter
    """
    s = np.sign(mn)
    lmin = np.log10(np.abs(mn)) if mn != 0 else 0
    lmax = np.log10(np.abs(mx)) if mx != 0 else 0
    ax, stp = np.linspace(s * lmin, lmax, num=num, retstep=True)
    sg = np.sign(ax)
    # val = sg * (10 ** np.abs(ax))
    val = 10 ** ax
    return ax, val

In [ ]:
# should validate mn << md << mx
# should distinguish between 2 cases where mn and mx have different signs or not

In [63]:
num = 20

In [104]:
np.log10(np.abs(100.0001))

np.float64(2.0000004342942646)

In [203]:
a = [-100,0.0001,1000]

smin, smid, smax = np.sign(a)
smin, smid, smax 

(np.float64(-1.0), np.float64(1.0), np.float64(1.0))

In [204]:
lmin, lmid, lmax =  np.log10(np.abs(a)) if a != 0 else 0
lmin, lmid, lmax 

(np.float64(2.0), np.float64(-4.0), np.float64(3.0))

In [221]:
axmin, stp = np.linspace( lmin, lmid, num=num+1, retstep=True)
axmin, stp

(array([ 2.000000,  1.700000,  1.400000,  1.100000,  0.800000,  0.500000,
         0.200000, -0.100000, -0.400000, -0.700000, -1.000000, -1.300000,
        -1.600000, -1.900000, -2.200000, -2.500000, -2.800000, -3.100000,
        -3.400000, -3.700000, -4.000000]),
 np.float64(-0.3))

In [222]:
axmin = axmin[:-1]
axmin

array([ 2.000000,  1.700000,  1.400000,  1.100000,  0.800000,  0.500000,
        0.200000, -0.100000, -0.400000, -0.700000, -1.000000, -1.300000,
       -1.600000, -1.900000, -2.200000, -2.500000, -2.800000, -3.100000,
       -3.400000, -3.700000])

In [224]:
valmin =smin * 10 ** axmin
valmin

array([-100.000000,  -50.118723,  -25.118864,  -12.589254,   -6.309573,
         -3.162278,   -1.584893,   -0.794328,   -0.398107,   -0.199526,
         -0.100000,   -0.050119,   -0.025119,   -0.012589,   -0.006310,
         -0.003162,   -0.001585,   -0.000794,   -0.000398,   -0.000200])

array([-6.000000, -5.700000, -5.400000, -5.100000, -4.800000, -4.500000,
       -4.200000, -3.900000, -3.600000, -3.300000, -3.000000, -2.700000,
       -2.400000, -2.100000, -1.800000, -1.500000, -1.200000, -0.900000,
       -0.600000, -0.300000])

In [208]:
axmax, stp = np.linspace(lmid, lmax, num=num+1, retstep=True) #, endpoint=False)
axmax, stp

(array([-4.000000, -3.650000, -3.300000, -2.950000, -2.600000, -2.250000,
        -1.900000, -1.550000, -1.200000, -0.850000, -0.500000, -0.150000,
         0.200000,  0.550000,  0.900000,  1.250000,  1.600000,  1.950000,
         2.300000,  2.650000,  3.000000]),
 np.float64(0.35))

In [209]:
axmax = axmax[1:]
axmax


array([-3.650000, -3.300000, -2.950000, -2.600000, -2.250000, -1.900000,
       -1.550000, -1.200000, -0.850000, -0.500000, -0.150000,  0.200000,
        0.550000,  0.900000,  1.250000,  1.600000,  1.950000,  2.300000,
        2.650000,  3.000000])

In [210]:
valmax = smax * 10 ** axmax
valmax

array([   0.000224,    0.000501,    0.001122,    0.002512,    0.005623,
          0.012589,    0.028184,    0.063096,    0.141254,    0.316228,
          0.707946,    1.584893,    3.548134,    7.943282,   17.782794,
         39.810717,   89.125094,  199.526231,  446.683592, 1000.000000])

In [226]:
val = np.concat([valmin,valmax])

In [227]:
ax = np.concat([-axmin - (-lmid), -lmid + axmax])

In [228]:
ax

array([-6.000000, -5.700000, -5.400000, -5.100000, -4.800000, -4.500000,
       -4.200000, -3.900000, -3.600000, -3.300000, -3.000000, -2.700000,
       -2.400000, -2.100000, -1.800000, -1.500000, -1.200000, -0.900000,
       -0.600000, -0.300000,  0.350000,  0.700000,  1.050000,  1.400000,
        1.750000,  2.100000,  2.450000,  2.800000,  3.150000,  3.500000,
        3.850000,  4.200000,  4.550000,  4.900000,  5.250000,  5.600000,
        5.950000,  6.300000,  6.650000,  7.000000])

In [229]:
len( ax)

40

In [230]:
px.scatter(y=val, x=ax, labels = {"x":"ax", "y":"val"}, width=1000)

In [ ]:
smin, smid, smax = np.sign(a)
smin, smid, smax 
lmin, lmid, lmax = np.log10(np.abs(a)) if a != 0 else 0
lmin, lmid, lmax 
axmin, stp = np.linspace( lmin, lmid, num=num, retstep=True)
axmin, stp
axmin = axmin[:-1]
axmin
valmin = smin * 10 ** axmin
valmin
axmax, stp = np.linspace(lmid, lmax, num=num+1, retstep=True) #, endpoint=False)
axmax, stp
axmax = axmax[1:]
axmax
valmax = smax * 10 ** axmax
valmax
val = np.concat([valmin,valmax])
ax = np.concat([-lmin - axmin, -lmid + axmax])

(np.float64(-2.0), np.float64(-2.0))

In [ ]:
def exp_scale(min, mid, max, num):
    """Produces an exponentially scaled axis for the slider control.
        When you need to control a variable with possible values in a wide range of orders of magnitude.
        e.g.  0.0001 < x < 10000

    Args:
        min (float): minimal value of the scale. also defines maximal order of magnitude in negative direction
        mid (float): defines minimal order of magnitude in both directions
        max (float): maximal value of the scale. also defines maximal order of magnitude in positive direction
        num (int): the amount of marks to produce
        

    Returns:
        float, float: ax - the linear values for the slider; val - the exponential values for the parameter
    """
    assert(min < mid < max)
    assert(min <  0  < max)
    a = np.array([min, mid, max])
    num = int(num/2)
    smin, smid, smax = np.sign(a)
    lmin, lmid, lmax = np.log10(np.abs(a))#if a != 0 else 0
    axmin = np.linspace( lmin, lmid, num=num+1)[:-1]
    axmax = np.linspace( lmid, lmax, num=num+1)[1:]
    valmin = smin * 10 ** axmin
    valmax = smax * 10 ** axmax
    val = np.concat([valmin,valmax])
    ax  = np.concat([-axmin - (-lmid), -lmid + axmax])
    # lbl = np.concat([np.zeros_like(valmin), np.ones_like(valmax)]).astype(bool)
    return ax, val#, lbl




In [293]:
a = [-150,-0.001,428]
ax,val = exp_scale(*a, 60)

In [294]:
ax.shape

(60,)

In [295]:
px.scatter(y=val, x=ax, color = lbl, labels = {"x":"ax", "y":"val"}, width=1000)

In [13]:
ax, val = exp_scale(-100, 100, 60)
